In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [3]:
df = pd.read_csv('dat_train1.csv')

In [4]:
df.head()

,customer_id,account_id,ed_id,event_name,event_timestamp,journey_steps_until_end,id,sep
0,15849251,383997507,4,browse_products,2021-11-04T14:11:15Z,1,15849251 383997507,-
1,15849251,383997507,4,browse_products,2021-11-04T14:11:29Z,2,15849251 383997507,-
2,15849251,383997507,4,browse_products,2021-11-04T14:12:10Z,3,15849251 383997507,-
3,15849251,383997507,4,browse_products,2021-11-04T14:12:21Z,4,15849251 383997507,-
4,15849251,383997507,4,browse_products,2021-11-04T14:12:24Z,5,15849251 383997507,-


In [5]:
df.shape

(54960961, 8)

In [6]:
len(list(df['customer_id'].unique()))

1391421

In [7]:
df['event_timestamp'] = pd.to_datetime(df['event_timestamp'])

In [8]:
df['event_timestamp'].min()

Timestamp('2020-11-03 03:31:30+0000', tz='UTC')

In [9]:
df['event_timestamp'].max()

Timestamp('2023-01-23 12:29:56+0000', tz='UTC')

## Task 1:
### 1. How many rows does your data set have?
    54960961
### 2. How many unqiue Ids are there?
    1391421
### 3. What is the earliest and latest time stamp?
    earliest: Timestamp('2020-11-03 03:31:30+0000', tz='UTC')
    lastest: Timestamp('2023-01-23 12:29:56+0000', tz='UTC')


In [10]:
df.columns

Index(['customer_id', 'account_id', 'ed_id', 'event_name', 'event_timestamp',
       'journey_steps_until_end', 'id', 'sep'],
      dtype='object')

In [11]:
df[['customer_id', 'account_id', 'ed_id', 'event_name', 'event_timestamp']].duplicated().sum()

np.int64(3112100)

In [12]:
df[['customer_id', 'account_id', 'ed_id', 'event_name', 'event_timestamp']].duplicated().mean()

np.float64(0.056623827956720045)

In [13]:
df = df.drop_duplicates(subset=['customer_id', 'account_id', 'ed_id', 'event_name', 'event_timestamp']).copy()

In [14]:
df.shape

(51848861, 8)

## Task 2

### 1. Duplicate Entries
There are **3112100 duplicate entries** in the dataset, which represents **5.66%** of all rows.

### 2. Dataset Size After Removing Duplicates
After removing duplicates, the dataset contains **51848861**.


In [15]:
df.head()

,customer_id,account_id,ed_id,event_name,event_timestamp,journey_steps_until_end,id,sep
0,15849251,383997507,4,browse_products,2021-11-04 14:11:15+00:00,1,15849251 383997507,-
1,15849251,383997507,4,browse_products,2021-11-04 14:11:29+00:00,2,15849251 383997507,-
2,15849251,383997507,4,browse_products,2021-11-04 14:12:10+00:00,3,15849251 383997507,-
3,15849251,383997507,4,browse_products,2021-11-04 14:12:21+00:00,4,15849251 383997507,-
4,15849251,383997507,4,browse_products,2021-11-04 14:12:24+00:00,5,15849251 383997507,-


In [37]:
task3_df = df.sample(frac=0.10, random_state=123)

In [45]:
task3_df.shape

(5184886, 8)

In [46]:
task3_df.head()

,customer_id,account_id,ed_id,event_name,event_timestamp,journey_steps_until_end,id,sep
50497354,-2114996295,-1758791062,6,begin_checkout,2022-08-20 22:32:18+00:00,45,-2114996295 -1758791062,-
31263134,-1043163780,-822729187,4,browse_products,2021-02-25 13:35:51+00:00,65,-1043163780 -822729187,-
48192184,-1916011503,1015681513,4,browse_products,2022-01-13 23:12:33+00:00,214,-1916011503 1015681513,-
18831360,-1514799169,1654112471,12,application_web_approved,2021-02-01 13:59:10+00:00,7,-1514799169 1654112471,-
12827062,1419546879,-1123903460,4,browse_products,2021-07-11 22:12:41+00:00,6,1419546879 -1123903460,-


In [38]:
journeys = (
    task3_df.sort_values(["customer_id", "account_id", "event_timestamp"])
      .groupby(["customer_id", "account_id"], as_index=False)
      .agg(
          event_names=("event_name", list),
          event_timestamps=("event_timestamp", list),
          max_journey_steps_until_end=("journey_steps_until_end", "max")
      )
)


In [ ]:
durations_sec = journeys['event_timestamps'].apply(
    lambda ts: (ts[-1] - ts[0]).total_seconds() if len(ts) >= 2 else np.nan
)

avg_duration = pd.to_timedelta(durations_sec.mean(), unit='s')
avg_duration

Timedelta('78 days 12:46:44.540645648')

In [40]:
journeys['max_journey_steps_until_end'].mean()

np.float64(36.5591340935136)

In [ ]:
value_counts = pd.DataFrame(task3_df['event_name'].value_counts())
value_counts.head()

,count
event_name,
browse_products,1942011
view_cart,594798
application_web_view,586474
promotion_created,557940
add_to_cart,369209


In [50]:
event_defintions = pd.read_csv('Event Definitions.csv')
event_defintions.head()

,event_name,journey_id,event_definition_id,milestone_number,stage
0,application_phone_approved,1,15,1.0,Apply for Credit
1,application_phone_declined,1,16,NaN,Apply for Credit
2,application_phone_pending,1,17,NaN,Apply for Credit
3,application_web_approved,1,12,1.0,Apply for Credit
4,application_web_declined,1,13,NaN,Apply for Credit


In [63]:
merged = value_counts.merge(event_defintions, on='event_name', how='left')

In [64]:
merged.head()

,event_name,count,journey_id,event_definition_id,milestone_number,stage
0,browse_products,1942011,1.0,4.0,NaN,First Purchase
1,view_cart,594798,1.0,5.0,NaN,First Purchase
2,application_web_view,586474,1.0,19.0,NaN,Apply for Credit
3,promotion_created,557940,NaN,NaN,NaN,NaN
4,add_to_cart,369209,1.0,11.0,NaN,First Purchase


In [67]:
gaps_sec = journeys['event_timestamps'].apply(
    lambda ts: [(ts[i] - ts[i-1]).total_seconds() for i in range(1, len(ts))]
    if len(ts) >= 2 else []
).explode().astype(float)

avg_gap = pd.to_timedelta(gaps_sec.mean(), unit='s')
avg_gap


Timedelta('18 days 02:05:53.115288429')

## Task 3.

### 1. How long are typical journeys? Actual time and actions.

**78 days, and around 36-37 actions**

### 2. What are common actions? Link the action IDs with the actual action descriptions to answer this question.

the top 5 most commont actions are

```text
1. browse_products          id = 4.0
2. view_cart                id = 5.0
3. application_web_view     id = 19.0
4. promotion_created        id = NaN
5. add_to_cart              id = 11.0
```

### 3. How much time is between an individual user’s actions?

the average gap between events in 18 days
